In [1]:
pip install transformers torch pandas accelerate sentencepiece

    torch (>=1.7.*)
           ~~~~~~^
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Model Loaded Successfully")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model Loaded Successfully


In [3]:
df = pd.read_excel("Allahabad_criminal_merged.xlsx")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (199, 10)


,case_id,language,case_type,judgment_text,subject,object,objective_aspect,subjective_aspect,reasoning,legal provision
0,ALL_CRIM_000001,Hindi,Criminal,Court No.=27\n\nAPPLICATION UIS 482 No. - 741 ...,Dinesh Yadav @ Dinesh Kumar And 7 Others,Wife/Complainant,Assault; Bigamy; Cruelty; Intentional Insult; ...,Dowry Demand,NaN,NaN
1,ALL_CRIM_000002,Hindi,Criminal,‘Neutral Citation No. - 2023:AHC:145371\n\nCou...,Kamlesh Jaiswal Alias Monu Jaiswal And Another,State/Society,Criminal Law Amendment Act Offence,Not Clearly Specified,NaN,NaN
2,ALL_CRIM_000003,Hindi,Criminal,‘Neutral Citation No. - 2024:AHC-LKO:1629\nCou...,Bahal Khan And 4 Others,Complainant/Informant,Rioting; Rioting With Deadly Weapon; Assault; ...,Intent/Knowledge To Cause Death Or Serious Harm,NaN,NaN
3,ALL_CRIM_000004,Hindi,Criminal,"Court No, -27\n(Case := APPLICATION U/S 482 No...",Banshi Lal,Complainant/Informant,Kidnapping Or Abduction,Intent To Compel Wrongful Restraint/Confinement,NaN,NaN
4,ALL_CRIM_000005,Hindi,Criminal,Neutral Citation No. - 2024:AHC-LKO:7688\nCour...,Mohd. Mahfooz,Wife/Complainant,Assault; Mischief; Cruelty; Intentional Insult,Dowry Demand,NaN,NaN


In [4]:
sample_df = df.head(5).copy()

print("Records Selected:", len(sample_df))

Records Selected: 5


In [19]:
sample_df
len(sample_df.loc[0, "judgment_text"])

2275

In [11]:
def build_prompt(row):

    return f"""
You are an expert Indian legal reasoning assistant.

Your task is to generate a concise explanation of the court's final reasoning and outcome based only on the judgment.

Judgment:
{row['judgment_text']}

Instructions:
1. Focus primarily on the court's final reasoning and outcome.
2. Explain why the court allowed, dismissed, quashed, granted bail, rejected bail, convicted, acquitted, or otherwise disposed of the case.
3. Base the explanation only on the reasoning contained in the judgment.
4. Do not merely restate allegations from the FIR or procedural history unless they are necessary to explain the court's reasoning.
5. Do not hallucinate or introduce legal facts not present in the judgment.
6. Write 4-8 clear and coherent sentences.

Output only the reasoning text.
"""

In [12]:
prompt = build_prompt(sample_df.iloc[0])

print(prompt[:3000])


You are an expert Indian legal reasoning assistant.

Your task is to generate a concise explanation of the court's final reasoning and outcome based only on the judgment.

Judgment:
Court No.=27

APPLICATION UIS 482 No. - 741 of 2024

 

 

Applicant :- Dinesh Yadav @ Dinesh Kumar And 7 Others
Opposite Party :- State Of U.P. Thru, Prin, Secy. Home Deptt. Lko. And
Another

Counsel for Applicant
Yadav

Counsel for Opposite Party

 

Santosh Kumar Srivastava Subhash Chandra

GAL

 

‘Hon‘ble Subhash Vidyarthi,J.

1. प्रार्थीगण के विद्वान अधिवक्ता श्री संतोष कुमार श्रीवास्तव , विद्वान अतिरिक्त
शासकीय अधिवक्ता श्री राकेश कुमार सिंह तथा विपक्षी संख्या 2 के विद्वान अधिवक्ता
श्री सुभाष चन्द्र यादव को सुना तथा पत्रावली का अवलोकन किया।

 

2. धारा 482 दण्ड प्रक्रिया संहिता के अन्तर्गत प्रस्तुत इस प्रार्थना पत्र द्वार प्राथीगण ने
प्रथम सूचना रिपोर्ट संख्या 451 सन 2019 अन्तर्गत RT 923, 494, 498-A,
504, 506 भाठदं०सं० थाना विभूति खण्ड जनपद लखनऊ के अमुक्रम में प्रस्तुत
आरोप पत्र दिनांकित 16.6.2020 ज

In [13]:
row = sample_df.iloc[0]

prompt = build_prompt(row)

messages = [
    {
        "role": "system",
        "content": "You are a legal reasoning expert."
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.3,
    top_p=0.9,
    do_sample=True
)

response = tokenizer.decode(
    outputs[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print(response)

The court allowed the application under Section 482 CrPC as the parties had reached a settlement through a reconciliation deed, which was affirmed by the trial court on December 7, 2023. Consequently, the prosecution against the applicants under Sections 323, 494, 498A, 504, and 506 of the Indian Penal Code in Case No. 15493/2021 before the Principal Magistrate, Lucknow, was quashed.


In [14]:
generated_reasonings = []

for idx, row in sample_df.iterrows():

    prompt = build_prompt(row)

    messages = [
        {
            "role": "system",
            "content": "You are a legal reasoning expert."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.3,
        top_p=0.9,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    generated_reasonings.append(response)

    print("="*100)
    print(f"CASE {idx}")
    print(response)
    print()

CASE 0
The court allowed the application under Section 482 CrPC as the parties had reached a mutual settlement as evidenced by the reconciliation deed which was affirmed by the judicial magistrate. The court dismissed the criminal charges against the applicants based on the reconciliation deed and the subsequent affirmation by the judicial magistrate.

CASE 1
The court allowed the anticipatory bail application after considering the arguments presented by the learned advocate for the applicant. The court found that the accused had not committed any offense as alleged and was not guilty of the charges framed against him. The court also noted that the accused had no criminal history and no interest against him in the case. Consequently, the court directed that if the applicant surrenders and submits to bail, his bail application would be considered by the High Court in accordance with the procedure laid down in Satendra Kumar Antil vs. Central Bureau of Investigation and another (Special 

In [16]:
sample_df["reasoning"] = generated_reasonings

sample_df[
    [
        "case_id",
        "subject",
        "object",
        "reasoning"
    ]
].head()

,case_id,subject,object,reasoning
0,ALL_CRIM_000001,Dinesh Yadav @ Dinesh Kumar And 7 Others,Wife/Complainant,The court allowed the application under Sectio...
1,ALL_CRIM_000002,Kamlesh Jaiswal Alias Monu Jaiswal And Another,State/Society,The court allowed the anticipatory bail applic...
2,ALL_CRIM_000003,Bahal Khan And 4 Others,Complainant/Informant,The court allowed the application under Sectio...
3,ALL_CRIM_000004,Banshi Lal,Complainant/Informant,"Based on the evidence and arguments presented,..."
4,ALL_CRIM_000005,Mohd. Mahfooz,Wife/Complainant,The court allowed the application under Sectio...


In [17]:
sample_df.to_csv(
    "Allahabad_criminal_reasoning_sample.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully
